# Exploración de SpaPhish

Revisamos el dataset limpio (`ml/data/processed/correos.csv`, generado por `ml/sicae_ml/load_data.py`)
antes de entrenar el modelo **phishing vs legitimo**.

Las gráficas se guardan como PNG en `ml/reports/`.

## 0. Configuración

Carga de librerías, rutas y datos. Colores del proyecto: **azul = legitimo**, **naranja = phishing**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Carpeta ml/, tanto si el notebook se abre desde ml/notebooks/ como desde la raíz del proyecto.
CARPETA_ML = Path.cwd().resolve()
if CARPETA_ML.name == "notebooks":
    CARPETA_ML = CARPETA_ML.parent
elif (CARPETA_ML / "ml").is_dir():
    CARPETA_ML = CARPETA_ML / "ml"

from sicae_ml.texto import palabras_frecuentes

RUTA_DATOS = CARPETA_ML / "data" / "processed" / "correos.csv"
CARPETA_REPORTES = CARPETA_ML / "reports"
CARPETA_REPORTES.mkdir(exist_ok=True)

COLORES = {"legitimo": "#2563EB", "phishing": "#F97316"}  # azul, naranja
ORDEN = ["legitimo", "phishing"]

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

df = pd.read_csv(RUTA_DATOS, sep=",", encoding="utf-8-sig")
print(f"{len(df)} correos y {df.shape[1]} columnas: {', '.join(df.columns)}")

## 1. Correos por etiqueta

¿Cuántos correos hay de cada clase? Si una clase fuera mucho más grande, el modelo tendería a
favorecerla.

In [ ]:
conteo = df["etiqueta"].value_counts().reindex(ORDEN)
tabla = pd.DataFrame({"correos": conteo, "porcentaje": (conteo / conteo.sum() * 100).round(1)})
display(tabla)

fig, ax = plt.subplots(figsize=(5, 3.5))
barras = ax.bar(conteo.index, conteo.values, color=[COLORES[e] for e in conteo.index])
ax.bar_label(barras)
ax.set_title("Correos por etiqueta")
ax.set_ylabel("Correos")
fig.tight_layout()
fig.savefig(CARPETA_REPORTES / "01_conteo_etiquetas.png")
plt.show()

## 2. Longitud del cuerpo por etiqueta

Número de caracteres del cuerpo: promedio, mediana y su distribución. El histograma se corta en el
percentil 99 para que unos pocos correos larguísimos no aplasten la gráfica.

In [ ]:
df["long_cuerpo"] = df["cuerpo"].str.len()
display(df.groupby("etiqueta")["long_cuerpo"].agg(promedio="mean", mediana="median").reindex(ORDEN).round(0))

limite = df["long_cuerpo"].quantile(0.99)
fig, ax = plt.subplots(figsize=(7, 4))
for etiqueta in ORDEN:
    datos = df.loc[df["etiqueta"] == etiqueta, "long_cuerpo"].clip(upper=limite)
    ax.hist(datos, bins=40, alpha=0.6, color=COLORES[etiqueta], label=etiqueta)
ax.set_title(f"Longitud del cuerpo (cortado en el percentil 99 = {limite:,.0f} caracteres)")
ax.set_xlabel("Caracteres")
ax.set_ylabel("Correos")
ax.legend()
fig.tight_layout()
fig.savefig(CARPETA_REPORTES / "02_longitud_cuerpo.png")
plt.show()

## 3. Enlaces (`n_urls`) y adjuntos (`n_adjuntos`) por etiqueta

Promedio, mediana, máximo y porcentaje de correos con al menos uno.

In [ ]:
resumen = df.groupby("etiqueta")[["n_urls", "n_adjuntos"]].agg(
    ["mean", "median", "max", lambda s: (s > 0).mean() * 100]
).reindex(ORDEN)
resumen.columns = [f"{col}_{est}" for col, est in zip(
    resumen.columns.get_level_values(0),
    ["promedio", "mediana", "maximo", "%_con_al_menos_1"] * 2,
)]
display(resumen.round(2).T)

fig, ejes = plt.subplots(1, 2, figsize=(9, 4))
for ax, columna in zip(ejes, ["n_urls", "n_adjuntos"]):
    grupos = [df.loc[df["etiqueta"] == e, columna] for e in ORDEN]
    caja = ax.boxplot(grupos, tick_labels=ORDEN, patch_artist=True, showfliers=False)
    for parche, etiqueta in zip(caja["boxes"], ORDEN):
        parche.set_facecolor(COLORES[etiqueta])
        parche.set_alpha(0.6)
    ax.set_title(columna)
fig.suptitle("Enlaces y adjuntos por etiqueta (sin valores atípicos)")
fig.tight_layout()
fig.savefig(CARPETA_REPORTES / "03_urls_adjuntos.png")
plt.show()

## 4. Las 20 palabras más frecuentes en phishing y en legitimo

Se usan asunto y cuerpo, en minúsculas, sin URLs, sin palabras de menos de 3 letras y sin
palabras vacías en español (`STOPWORDS_ES` de `ml/sicae_ml/texto.py`).

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 6))
for ax, etiqueta in zip(ejes, ["phishing", "legitimo"]):
    sub = df[df["etiqueta"] == etiqueta]
    top = palabras_frecuentes(sub["asunto"] + " " + sub["cuerpo"], n=20)
    palabras, veces = zip(*top)
    ax.barh(palabras[::-1], veces[::-1], color=COLORES[etiqueta])
    ax.set_title(f"{etiqueta} ({len(sub)} correos)")
    ax.set_xlabel("Apariciones")
    print(f"{etiqueta}: {', '.join(palabras)}")
fig.suptitle("Top 20 palabras por etiqueta")
fig.tight_layout()
fig.savefig(CARPETA_REPORTES / "04_palabras_frecuentes.png")
plt.show()

## 5. Dos ejemplos de cada clase

Muestra fija (`random_state=42`) para que siempre salgan los mismos. El cuerpo se recorta a 300
caracteres.

In [ ]:
def recortar(texto, limite=300):
    texto = " ".join(str(texto).split())  # quita saltos de línea y espacios repetidos
    return texto if len(texto) <= limite else texto[:limite] + "…"


for etiqueta in ["phishing", "legitimo"]:
    print(f"========== {etiqueta.upper()} ==========")
    for _, fila in df[df["etiqueta"] == etiqueta].sample(2, random_state=42).iterrows():
        print(f"Asunto: {fila['asunto']}")
        print(f"Cuerpo: {recortar(fila['cuerpo'])}\n")

## Conclusiones

1. **Clases balanceadas** (52,3 % phishing, 47,7 % legitimo): no hace falta rebalancear; basta
   con dividir train/test de forma estratificada.
2. **El phishing es más corto**: mediana de 539 caracteres frente a 1.115 en legitimo. La longitud
   puede ser una señal útil.
3. **Enlaces y adjuntos no se comportan como se espera**: casi todo el phishing trae al menos un
   enlace (96 %), pero los legítimos traen *más* enlaces en promedio (8,5 frente a 5,0) y casi todos
   los adjuntos (38 % con adjunto frente a 3 %). Sirven como señales, pero no como regla simple
   "muchos enlaces = phishing".
4. **Vocabulario distinto**: el phishing habla de paquetes, entregas, pagos, suscripciones e iCloud
   (urgencia y suplantación de marcas); el legítimo, de información, datos personales, privacidad,
   instituciones y saludos.
5. **Riesgos para el modelo** (revisar en el preprocesamiento):
   - *pablito* aparece en 169 correos de phishing y en ninguno legítimo: es un nombre de
     destinatario, no una señal real de phishing. El modelo lo aprendería como atajo.
   - Lo mismo puede pasar con nombres de lugar o persona del lado legítimo (*puebla*, *ricardo*).
   - 210 asuntos de phishing están repetidos (campañas con la misma plantilla). Si copias de una
     campaña quedan en train y en test, la evaluación saldrá inflada: conviene agrupar por asunto
     al dividir.
   - Hay texto en inglés (*the*, *and*) que las palabras vacías en español no filtran.
   - *darse de baja* aparece en 187 correos de phishing: el phishing también imita a la
     publicidad, así que la regla de publicidad debe apoyarse en la cabecera `List-Unsubscribe`
     y no solo en el texto.
   - Algunos correos marcados *legitimo* son promociones (por ejemplo, el ejemplo de BetterMe):
     en SpaPhish la publicidad cuenta como legítima, y nuestra categoría *publicidad* la separa
     la regla, no el modelo.